# PBIT Data Inspection

This notebook validates the long/tidy PBIT dataset used as the base for feature engineering and modeling.

## Setup

The notebook reuses the same validation code as `scripts/01_validate_data.py` so the exploratory view and script output remain consistent.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pbit.data_validation import (
    assert_expected_long_dataset,
    load_long_dataset,
    summarize_long_dataset,
)

DATA_PATH = PROJECT_ROOT / "data" / "PBIT_atuador_DESCARAC_r1_long.csv"
DATA_PATH

## Load Dataset

In [ ]:
df = load_long_dataset(DATA_PATH)
df.head(10)

In [ ]:
summary = summarize_long_dataset(df)
assert_expected_long_dataset(summary)

{
    "rows": summary.rows,
    "columns": summary.columns,
    "experiments": summary.experiments,
    "serial_numbers": summary.serial_numbers,
    "sample_positions": summary.sample_positions,
    "rows_per_experiment": summary.rows_per_experiment,
    "missing_measurements": summary.missing_measurements,
}

## Label Distribution

In [ ]:
experiment_meta = (
    df[["Experiment_Index", "Serial_Number", "Serial_Run_Index", "Fault_Mode", "Severity_Index"]]
    .drop_duplicates("Experiment_Index")
    .sort_values("Experiment_Index")
)

fault_counts = experiment_meta["Fault_Mode"].value_counts().sort_index().rename("Tests")
fault_distribution = fault_counts.to_frame()
fault_distribution["Share"] = fault_distribution["Tests"] / fault_distribution["Tests"].sum()
fault_distribution

In [ ]:
severity_counts = experiment_meta["Severity_Index"].value_counts().sort_index().rename("Tests")
severity_distribution = severity_counts.to_frame()
severity_distribution["Share"] = severity_distribution["Tests"] / severity_distribution["Tests"].sum()
severity_distribution

## F06 Coverage

F06 is the most constrained class. These records and serial numbers should be tracked in every split diagnostic.

In [ ]:
summary.f06_records

In [ ]:
summary.f06_records["Serial_Number"].value_counts().sort_index()

## Missing Measurements

The original Excel `#VALUE!` cells were normalized to numeric missing values in the long CSV.

In [ ]:
summary.bad_records

## One Example Cycle

Each `Experiment_Index` contains 123 ordered measurements. `Sample_Index` is the time-like axis for the PBIT movement.

In [ ]:
example = df.loc[df["Experiment_Index"] == 1111, ["Sample_Index", "Measurement"]]
ax = example.plot(x="Sample_Index", y="Measurement", legend=False, figsize=(9, 3), title="Experiment 1111 PBIT Signal")
ax.set_xlabel("Sample Index")
ax.set_ylabel("Measurement")

---
## Signal Quality Classification

Missing values fall into two distinct categories that require different treatment:

| Quality | Criterion | Action |
|---------|-----------|--------|
| `sparse` | ≤ 50 % of samples missing | Linear interpolation within the experiment |
| `corrupt` | > 50 % of samples missing | Exclude from dataset; document separately |

Interpolation is intra-experiment (uses only values from the same PBIT cycle) and
therefore does not leak information across train/validation folds.

In [ ]:
from pbit.preprocessing import classify_missing

quality = classify_missing(df)
print(quality["Quality"].value_counts().to_string())
print()
quality[quality["Quality"] != "ok"]

## Preprocessing

Apply both steps — interpolation then exclusion — and inspect the results.

In [ ]:
from pbit.preprocessing import preprocess

df_clean, excluded = preprocess(df)

print("=== Excluded experiments (corrupt signal) ===")
print(excluded.to_string(index=False))
print()
print(f"Raw dataset      : {df['Experiment_Index'].nunique():,} experiments, {len(df):,} rows")
print(f"Clean dataset    : {df_clean['Experiment_Index'].nunique():,} experiments, {len(df_clean):,} rows")
print(f"Excluded         : {len(excluded)} experiments")
print(f"Remaining NaNs   : {df_clean['Measurement'].isna().sum()}")

### Interpolation check

Visual confirmation that the interpolated gap is plausible for one of the sparse experiments.

In [ ]:
import matplotlib.pyplot as plt

# Pick the first sparse experiment for comparison
from pbit.preprocessing import classify_missing
sparse_idx = classify_missing(df).query("Quality == 'sparse'")["Experiment_Index"].iloc[0]

raw_sig   = df.loc[df["Experiment_Index"] == sparse_idx, ["Sample_Index", "Measurement"]].sort_values("Sample_Index")
clean_sig = df_clean.loc[df_clean["Experiment_Index"] == sparse_idx, ["Sample_Index", "Measurement"]].sort_values("Sample_Index")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(clean_sig["Sample_Index"], clean_sig["Measurement"], label="interpolated", color="steelblue")
missing_mask = raw_sig["Measurement"].isna()
ax.scatter(
    raw_sig.loc[missing_mask, "Sample_Index"],
    clean_sig.loc[missing_mask, "Measurement"],
    color="red", zorder=5, label="filled gap", s=60
)
ax.set_title(f"Experiment {sparse_idx} — interpolated gap (red dot)")
ax.set_xlabel("Sample Index")
ax.set_ylabel("Measurement")
ax.legend()
plt.tight_layout()
plt.show()

### Class distribution after preprocessing

F06 loses 2 of its 9 experiments (the corrupt ones from SN 045). This must be reported explicitly.

In [ ]:
import pandas as pd

meta_clean = (
    df_clean[["Experiment_Index", "Fault_Mode", "Severity_Index"]]
    .drop_duplicates("Experiment_Index")
)

before = experiment_meta["Fault_Mode"].value_counts().sort_index().rename("Before")
after  = meta_clean["Fault_Mode"].value_counts().sort_index().rename("After")

comparison = pd.concat([before, after], axis=1)
comparison["Diff"] = comparison["After"] - comparison["Before"]
comparison

## Save Clean Dataset

The clean long CSV (`PBIT_atuador_DESCARAC_r1_long_clean.csv`) is the input for
`notebooks/02_feature_engineering.ipynb`.  The raw file is kept unchanged.

In [ ]:
CLEAN_PATH = PROJECT_ROOT / "data" / "PBIT_atuador_DESCARAC_r1_long_clean.csv"
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved: {CLEAN_PATH}")
print(f"Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"Experiments: {df_clean['Experiment_Index'].nunique():,}")
print(f"Remaining NaNs: {df_clean['Measurement'].isna().sum()}")

## Per-Equipment Signal Normalisation

Z-score normalisation within each Serial_Number removes equipment-level DC offset
and scale differences. After normalisation, features capture fault *shape* rather
than absolute amplitude, improving generalisation to unseen equipment.

Because each Serial_Number belongs exclusively to either train or test, statistics
computed on the full dataset are leak-free.

In [ ]:
from pbit.preprocessing import normalize_by_serial_number

NORM_PATH = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_normalized.csv'

df_normalized = normalize_by_serial_number(df_clean, method='zscore')

sn_check = (
    df_normalized.groupby('Serial_Number')['Measurement']
    .agg(['mean', 'std'])
    .describe()
    .round(4)
)
print('Per-SN stats after normalisation (mean~0, std~1):')
print(sn_check)

df_normalized.to_csv(NORM_PATH, index=False)
print('Saved:', NORM_PATH)
print('Rows:', len(df_normalized), '  Experiments:', df_normalized['Experiment_Index'].nunique())